# Entropy vs. Generative Perplexity Tradeoff Curves (Grid Plot)

This notebook collects and plots the results from evaluation sweeps across different values of `p_nucleus` and `temperature` for various step counts ($T$).

In [ ]:
import json
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

In [ ]:
def extract_sweep_data(experiments_dir, algo, t_val, param_name, values, keys_to_extract=["generative_ppl", "entropy"]):
    """
    Extracts generation metrics across different parameter sweep values for a fixed T.
    """
    results = []
    base_dir = Path(experiments_dir)
    
    # Match files for base names or variations (e.g. duo/duo_base, smflm/smflm_lin)
    algo_variants = [algo]
    if algo == "duo":
        algo_variants.append("duo_base")
    elif algo == "smflm":
        algo_variants.append("smflm_lin")
        
    for val in values:
        row_data = {
            "algo": algo,
            "T": t_val,
            param_name: val
        }
        # Initialize metrics with None
        for key in keys_to_extract:
            row_data[key] = None
            
        # Search variants
        file_path = None
        for name in algo_variants:
            filename = f"{name}_T-{t_val}_{param_name}-{val}.json"
            temp_path = base_dir / filename
            if temp_path.exists():
                file_path = temp_path
                break
            # Try alternative parameter name format
            alt_param = "temperature" if param_name == "p_nucleus" else "p_nucleus"
            alt_filename = f"{name}_T-{t_val}_{alt_param}-{val}.json"
            temp_path = base_dir / alt_filename
            if temp_path.exists():
                file_path = temp_path
                break
                
        if file_path and file_path.exists():
            try: 
                with file_path.open('r') as f:
                    data = json.load(f)
                for key in keys_to_extract:
                    row_data[key] = data.get(key)
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                
        results.append(row_data)
        
    return pd.DataFrame(results)

In [ ]:
# Configure sweep directories, target steps (T), and parameter range
SWEEPS_DIR = "../outputs/sweeps"  # Update to target folder path (e.g. your cluster mount or local copy)
TARGET_TS = [32, 64, 128, 256, 512, 1024]                 # Step sizes to plot in the grid
P_NUCLEUS_VALUES = [0.8, 0.825, 0.85, 0.875, 0.9, 0.925, 0.95, 0.975, 1.0]

algos_to_plot = ["mdlm", "sedd", "duo", "smflm", "flm"]

In [ ]:
# Plotting: Gen PPL vs Entropy (Grid Plot across all specified T's)
cols = 3
rows = math.ceil(len(TARGET_TS) / cols)

fig, axes = plt.subplots(nrows=rows, ncols=cols, figsize=(6 * cols, 5 * rows), squeeze=False)
axes = axes.flatten()

legend_handles = {}

for idx, target_t in enumerate(TARGET_TS):
    ax = axes[idx]
    
    all_data_t = []
    for algo in algos_to_plot:
        # Extract the dynamic data for the specific T
        df = extract_sweep_data(SWEEPS_DIR, algo, target_t, "p_nucleus", P_NUCLEUS_VALUES)
        all_data_t.append(df)
        
    df_plot = pd.concat(all_data_t, ignore_index=True)
    df_plot.dropna(subset=["generative_ppl", "entropy"], inplace=True)

    if df_plot.empty:
        print(f"Warning: No sweep data found for T={target_t}.")
        continue
        
    # Plot active model sweeps in the exact order of algos_to_plot to preserve color cycles
    for algo in algos_to_plot:
        group = df_plot[df_plot["algo"] == algo]
        if group.empty:
            continue
            
        group_sorted = group.sort_values(by="entropy")
        display_label = algo.upper()
        
        line, = ax.plot(
            group_sorted["entropy"], 
            group_sorted["generative_ppl"], 
            marker='o', 
            linestyle='-', 
            linewidth=2,
            label=display_label
        )
        if display_label not in legend_handles:
            legend_handles[display_label] = line
 
    # Format subplot to place optimal region at top-right and crop outliers
    ax.set_ylim(bottom=250.0, top=10.0)  # Inverted Y-axis: 250 is bottom, 10 is top
    ax.set_xlim(left=4.5, right=6.0)     # Focus on quality entropy range
    ax.set_title(f"T = {target_t} Steps", fontsize=14, fontweight='bold')
    ax.set_xlabel("Sample Entropy", fontsize=12)
    ax.set_ylabel("Generative Perplexity", fontsize=12)
    ax.grid(True, which="both", linestyle='--', alpha=0.5)

 # Hide any unused subplots
for idx in range(len(TARGET_TS), len(axes)):
    fig.delaxes(axes[idx])

# Add unified legend
if legend_handles:
    fig.legend(
        legend_handles.values(), 
        legend_handles.keys(), 
        fontsize=11, 
        bbox_to_anchor=(1.02, 0.5), 
        loc='center left'
    )

plt.tight_layout()
output_file = "entropy_vs_perplexity_grid.png"
plt.savefig(output_file, dpi=300, bbox_inches='tight')
print(f"Grid plot saved successfully to {output_file}")
plt.show()